# 04 — MHW detection (mhw3d) and daily/monthly MHW fields

1. `common.calculate_mhw_metrics` on the daily severity (`minDuration=5`, `maxGap=2`), `maxEvt=cfg.MAX_EVT`.
2. Daily MHW state rebuilt from event start/end indices (so gaps ≤2 days that are joined into one
   event count as MHW days, as in Hobday 2016).
3. Monthly metrics: MHW days, days in category ≥2 (strong+, Hobday 2018), cumulative intensity
   (Σ T_anom on MHW days, °C·days), maximum intensity, event onsets.

Outputs: `mhw_events_*.nc`, `mhw_daily_*.zarr`, `mhw_monthly_*.nc`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
from mhw3d import common
client = cfg.start_client()
client

In [ ]:
sev = xr.open_zarr(cfg.F_SEV)
sev = sev.chunk({"time": -1, **cfg.CHUNKS_SPACE})
events = common.calculate_mhw_metrics(sev, maxEvt=cfg.MAX_EVT).compute()
nev = events.date_start.notnull().sum("event")
print("max events in a cell:", int(nev.max()), "/ maxEvt =", cfg.MAX_EVT)
assert int(nev.max()) < cfg.MAX_EVT, "Event slots saturated — increase cfg.MAX_EVT and rerun"
events = events.dropna("event", how="all")
events.attrs.update(baseline=str(cfg.BASELINE))
events.to_netcdf(cfg.F_EVENTS)
events

In [ ]:
def _event_mask(istart, iend, nt):
    d = np.zeros(nt + 1, dtype=np.int32)
    ok = np.isfinite(istart) & np.isfinite(iend)
    np.add.at(d, istart[ok].astype(int), 1)
    np.add.at(d, iend[ok].astype(int) + 1, -1)
    return (np.cumsum(d[:-1]) > 0).astype(np.int8)

nt = sev.sizes["time"]
ev = xr.open_dataset(cfg.F_EVENTS).chunk(cfg.CHUNKS_SPACE)
mhw = xr.apply_ufunc(_event_mask, ev.index_start, ev.index_end, kwargs=dict(nt=nt),
                     input_core_dims=[["event"], ["event"]], output_core_dims=[["time"]],
                     vectorize=True, dask="parallelized", output_dtypes=[np.int8],
                     dask_gufunc_kwargs=dict(output_sizes={"time": nt}))
mhw = mhw.assign_coords(time=sev.time).transpose("time", "lat", "lon")
mhw = mhw.where(sev.T_anom.isel(time=0).notnull().drop_vars("time"), 0).astype(np.int8)
xr.Dataset({"mhw": mhw}).chunk({"time": -1, **cfg.CHUNKS_SPACE}).to_zarr(cfg.F_MHW_DAILY, mode="w", consolidated=True)

In [ ]:
mhw = xr.open_zarr(cfg.F_MHW_DAILY).mhw.astype(bool)
ta = sev.T_anom
on = mhw & ~mhw.shift(time=1, fill_value=False)
daily = xr.Dataset(dict(
    mhw_days=mhw.astype("int16"),
    cat2_days=(mhw & (sev.severity >= 2)).astype("int16"),
    cum_int=ta.where(mhw, 0.0),
    max_int=ta.where(mhw),
    onsets=on.astype("int16"),
))
r = daily.resample(time="MS")
monthly = xr.Dataset(dict(mhw_days=r.sum().mhw_days, cat2_days=r.sum().cat2_days,
                          cum_int=r.sum().cum_int, max_int=daily.max_int.resample(time="MS").max(),
                          onsets=r.sum().onsets))
nd = ta.isel(lat=0, lon=0).resample(time="MS").count().compute()
complete = nd.time.values[(nd == nd.time.dt.days_in_month).values]
ocean = sev.T_anom.isel(time=0).notnull().drop_vars("time")
monthly = monthly.sel(time=complete).where(ocean).compute()
monthly.mhw_days.attrs["units"] = "days"; monthly.cat2_days.attrs["units"] = "days"
monthly.cum_int.attrs["units"] = "°C days"; monthly.max_int.attrs["units"] = "°C"
monthly.attrs.update(baseline=str(cfg.BASELINE))
monthly.to_netcdf(cfg.F_MHW_MONTHLY)
monthly

## Checks

In [ ]:
# consistency: MHW days from the daily mask vs Σ event durations
dur = events.duration.sum("event")
diff = float(abs(dur - mhw.sum("time")).max())
print("max |Σduration − Σmask| over cells:", diff)
masks = xr.open_dataset(cfg.F_MASKS)
ann = monthly.mhw_days.resample(time="YS").sum().where(masks.ocean)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
cfg.box_mean(ann, mask=masks.shelf).plot(ax=ax[0], marker="o"); ax[0].set_title("Shelf-mean MHW days / yr")
ann.mean("time").plot(ax=ax[1], cmap="magma"); ax[1].set_title("Mean MHW days / yr");